In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q --upgrade peft accelerate bitsandbytes

In [ ]:
import os, json, random, shutil, torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
 
# ============================================================
# CONFIG
# ============================================================
MODEL_ID = "Qwen/Qwen3.5-4B"
SFT_ADAPTER = "/kaggle/input/datasets/yuanmazax/delta-filing-sft-result"
DPO_DATA = "/kaggle/input/datasets/yuanmazax/delta-filing-dpo/dpo_pytorch.jsonl"
OUTPUT_BASE = "/kaggle/working/adapters"
 
BETA = 0.5
SFT_WEIGHT = 0.5
LR = 5e-5
EPOCHS = 1
MIN_STEPS_BEFORE_EARLY_STOP = 100  # Must train at least 100 steps
EARLY_STOP_LOSS = 0.005            # Tighter threshold
LABEL_SMOOTHING = 0.1
MAX_SEQ_LEN = 1024
LOG_EVERY = 10
EVAL_EVERY = 50
 
EXPERIMENTS = ["dpo", "ipo", "cdpo"]
 
SIMPLE_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<|im_start|>system\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}"
    "<|im_start|>user\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}"
    "<|im_start|>assistant\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)
 
SYSTEM_PROMPT = (
    "You are Delta Filing, a financial analyst AI specializing in SEC filing analysis. "
    "You analyze 10-K and 10-Q filings, detect year-over-year changes in risk disclosures, "
    "track management guidance accuracy, and flag potential red flags. "
    "Always reference specific filing sections, cite specific numbers and dates, "
    "provide analytical judgment, and note caveats. "
    "Do not give investment advice or predict stock prices."
)
TOOL_SYSTEM_PROMPT = (
    "You are Delta Filing, a financial analyst AI specializing in SEC filing analysis. "
    "You have access to tools for retrieving SEC filings, financial data, news, and insider trades. "
    "When you need data, call the appropriate tool. When you have data, analyze it thoroughly.\n\n"
    "Available tools:\n\n"
    "1. search_filings(ticker, filing_type, count)\n"
    "2. get_filing_section(ticker, section_id, filing_type)\n"
    "3. diff_filing_sections(ticker, section_id, filing_type)\n"
    "4. stock_price(ticker, period)\n"
    "5. company_metrics(ticker)\n"
    "6. company_news(ticker, days)\n"
    "7. insider_trades(ticker)\n"
    "8. analyst_ratings(ticker)\n\n"
    'To use a tool, respond with JSON: {"tool": "name", "arguments": {...}}\n'
    "Respond with ONLY the JSON, nothing else."
)
ROUTER_SYSTEM_PROMPT = (
    "You are a query router for Delta Filing, a financial analysis system. "
    "Classify the user's query into exactly one category. Respond with ONLY the category name.\n\n"
    "Categories:\n"
    "- FILING_ANALYSIS: Questions about a specific company's filing content\n"
    "- FILING_DIFF: Questions comparing filings across time periods\n"
    "- COMPANY_DEEP_DIVE: Requests for comprehensive company analysis\n"
    "- SIMPLE_QUERY: Simple factual queries about filings\n"
    "- OUT_OF_SCOPE: Questions unrelated to SEC filings"
)
REVIEW_SYSTEM_PROMPT = (
    "You are a quality reviewer for financial analysis. Review the analysis below and respond with either:\n"
    "- COMPLETE if the analysis is thorough with specific numbers, filing section references, and analytical judgment\n"
    "- NEEDS_FOLLOW_UP: <reason> if the analysis lacks depth, specifics, or analytical judgment"
)
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
 
print(f"GPU: {torch.cuda.get_device_name(0)}")
 
# ============================================================
# TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.chat_template = SIMPLE_CHAT_TEMPLATE
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
 
# ============================================================
# LOAD & TOKENIZE DPO DATA
# ============================================================
print("\n[1] Loading DPO data...")
with open(DPO_DATA) as f:
    raw_data = [json.loads(line) for line in f if line.strip()]
 
pairs = []
for ex in raw_data:
    cat = ex.get("category", "analysis")
    sp = TOOL_SYSTEM_PROMPT if cat == "tool_calling" else SYSTEM_PROMPT
    ct = tokenizer.apply_chat_template([
        {"role": "system", "content": sp},
        {"role": "user", "content": ex["question"]},
        {"role": "assistant", "content": ex["chosen"]},
    ], tokenize=False)
    rt = tokenizer.apply_chat_template([
        {"role": "system", "content": sp},
        {"role": "user", "content": ex["question"]},
        {"role": "assistant", "content": ex["rejected"]},
    ], tokenize=False)
    pairs.append({
        "chosen_ids": torch.tensor(tokenizer.encode(ct, max_length=MAX_SEQ_LEN, truncation=True)),
        "rejected_ids": torch.tensor(tokenizer.encode(rt, max_length=MAX_SEQ_LEN, truncation=True)),
    })
 
random.seed(42)
random.shuffle(pairs)
split = int(len(pairs) * 0.85)
train_pairs, valid_pairs = pairs[:split], pairs[split:]
print(f"  Train: {len(train_pairs)}, Valid: {len(valid_pairs)}")
 
 
# ============================================================
# CORE FUNCTIONS
# ============================================================
 
def compute_log_probs(model, input_ids):
    """Per-token AVERAGE log probability (length-normalized).
 
    FIX: Previous version summed log probs, biasing toward shorter sequences.
    Now we divide by sequence length to get a fair comparison regardless of length.
    """
    x = input_ids.unsqueeze(0).to("cuda:0")
    seq_len = x.shape[1]
 
    with torch.amp.autocast("cuda", dtype=torch.float16):
        out = model(x)
        logits = out.logits
 
    sl = logits[:, :-1, :].float()
    labels = x[:, 1:]
    lp = F.log_softmax(sl, dim=-1)
    tlp = torch.gather(lp, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
 
    # FIX: average over sequence length, not sum
    avg_log_prob = tlp.sum() / (seq_len - 1)
 
    del logits, sl, lp, tlp, out
    return avg_log_prob
 
 
def compute_sft_loss(model, chosen_ids):
    """Standard cross-entropy on chosen response."""
    x = chosen_ids.unsqueeze(0).to("cuda:0")
    with torch.amp.autocast("cuda", dtype=torch.float16):
        out = model(x, labels=x)
    loss = out.loss
    del out
    return loss
 
 
def preference_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta, loss_type, label_smoothing=0.1):
    """Compute preference loss — DPO, IPO, or cDPO.
 
    All inputs are now per-token average log probs (length-normalized).
    """
    chosen_reward = pi_chosen - ref_chosen
    rejected_reward = pi_rejected - ref_rejected
    margin = beta * (chosen_reward - rejected_reward)
 
    if loss_type == "dpo":
        loss = -F.logsigmoid(margin)
 
    elif loss_type == "ipo":
        target = 1.0 / (2.0 * beta)
        # FIX: clip margin to prevent gradient explosion from squaring large values
        margin_clipped = torch.clamp(margin, -10.0, 10.0)
        loss = (margin_clipped - target) ** 2
 
    elif loss_type == "cdpo":
        eps = label_smoothing
        loss = -(1 - eps) * F.logsigmoid(margin) - eps * F.logsigmoid(-margin)
 
    else:
        raise ValueError(f"Unknown loss type: {loss_type}")
 
    return loss, margin.item()
 
 
def combined_loss(model, chosen_ids, rejected_ids, ref_chosen, ref_rejected,
                  beta, loss_type, sft_weight, label_smoothing=0.1):
    """Combined preference + SFT retention loss."""
    pi_chosen = compute_log_probs(model, chosen_ids)
    pi_rejected = compute_log_probs(model, rejected_ids)
 
    pref_loss, margin = preference_loss(
        pi_chosen, pi_rejected, ref_chosen, ref_rejected,
        beta, loss_type, label_smoothing,
    )
 
    sft_loss = compute_sft_loss(model, chosen_ids)
 
    total = pref_loss + sft_weight * sft_loss
 
    return total, {
        "total": total.item(),
        "pref": pref_loss.item(),
        "sft": sft_loss.item(),
        "margin": margin,
    }
 
 
# ============================================================
# TRAINING FUNCTION
# ============================================================
 
def run_experiment(loss_type, train_pairs, valid_pairs, ref_cache_train, ref_cache_valid):
    print(f"\n{'='*60}")
    print(f"  EXPERIMENT: {loss_type.upper()}")
    print(f"  β={BETA}, sft_weight={SFT_WEIGHT}, epochs={EPOCHS}")
    print(f"{'='*60}")
 
    output_dir = os.path.join(OUTPUT_BASE, f"dpo_{loss_type}")
    os.makedirs(output_dir, exist_ok=True)
 
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        trust_remote_code=True, device_map={"": 0},
    )
    model = PeftModel.from_pretrained(base, SFT_ADAPTER, is_trainable=True)
    model.gradient_checkpointing_enable()
    model.train()
 
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=0.01,
    )
 
    step = 0
    best_val = float("inf")
    early_stopped = False
    indices = list(range(len(train_pairs)))
    pref_losses, margins = [], []
 
    for epoch in range(1, EPOCHS + 1):
        random.shuffle(indices)
        for i, idx in enumerate(indices):
            pair = train_pairs[idx]
            ref = ref_cache_train[idx]
 
            optimizer.zero_grad()
            loss, metrics = combined_loss(
                model, pair["chosen_ids"], pair["rejected_ids"],
                ref["ref_chosen"], ref["ref_rejected"],
                BETA, loss_type, SFT_WEIGHT, LABEL_SMOOTHING,
            )
 
            # Skip step if loss is NaN (safety)
            if torch.isnan(loss):
                print(f"  Step {step+1}: NaN loss, skipping")
                del loss
                torch.cuda.empty_cache()
                continue
 
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
 
            del loss
            torch.cuda.empty_cache()
 
            pref_losses.append(metrics["pref"])
            margins.append(metrics["margin"])
            step += 1
 
            if step % LOG_EVERY == 0:
                avg_l = sum(pref_losses[-LOG_EVERY:]) / LOG_EVERY
                avg_m = sum(margins[-LOG_EVERY:]) / LOG_EVERY
                print(f"  Step {step:4d} | pref={avg_l:.4f} | sft={metrics['sft']:.4f} | margin={avg_m:.3f}")
 
            if step % EVAL_EVERY == 0:
                model.eval()
                vl, vm = [], []
                with torch.no_grad():
                    for vi in range(min(20, len(valid_pairs))):
                        vp = valid_pairs[vi]
                        rv = ref_cache_valid[vi]
                        l, m = combined_loss(
                            model, vp["chosen_ids"], vp["rejected_ids"],
                            rv["ref_chosen"], rv["ref_rejected"],
                            BETA, loss_type, SFT_WEIGHT, LABEL_SMOOTHING,
                        )
                        if not torch.isnan(l):
                            vl.append(m["pref"])
                            vm.append(m["margin"])
                if vl:
                    avg_vl = sum(vl) / len(vl)
                    avg_vm = sum(vm) / len(vm)
                    is_best = avg_vl < best_val
                    if is_best:
                        best_val = avg_vl
                        model.save_pretrained(output_dir)
                        tokenizer.save_pretrained(output_dir)
                    print(f"  Step {step:4d} | VAL pref={avg_vl:.4f} | margin={avg_vm:.3f}"
                          f"{'  ★ saved' if is_best else ''}")
                model.train()
 
            # Early stopping: only after MIN_STEPS and with enough data
            if step >= MIN_STEPS_BEFORE_EARLY_STOP and len(pref_losses) >= 20:
                recent = sum(pref_losses[-20:]) / 20
                if recent < EARLY_STOP_LOSS and not any(
                    x != x for x in pref_losses[-20:]  # check no NaN
                ):
                    print(f"  Early stop at step {step}: avg pref loss {recent:.4f} < {EARLY_STOP_LOSS}")
                    early_stopped = True
                    break
 
        if early_stopped:
            break
 
    # Final save if no best was saved during eval
    if best_val == float("inf"):
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
 
    print(f"  {loss_type.upper()} done: {step} steps, best val pref={best_val:.4f}")
 
    del model, base, optimizer
    torch.cuda.empty_cache()
    return output_dir
 
 
# ============================================================
# PRE-COMPUTE REFERENCE LOG PROBS (length-normalized)
# ============================================================
 
print("\n[2] Pre-computing reference log probs (length-normalized)...")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    trust_remote_code=True, device_map={"": 0},
)
ref_model = PeftModel.from_pretrained(base, SFT_ADAPTER)
ref_model.eval()
 
ref_cache = []
with torch.no_grad():
    for i, pair in enumerate(pairs):
        rc = compute_log_probs(ref_model, pair["chosen_ids"]).item()
        rr = compute_log_probs(ref_model, pair["rejected_ids"]).item()
        ref_cache.append({"ref_chosen": rc, "ref_rejected": rr})
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(pairs)}")
            torch.cuda.empty_cache()
 
ref_train = ref_cache[:split]
ref_valid = ref_cache[split:]
avg_margin = sum(c["ref_chosen"] - c["ref_rejected"] for c in ref_cache[:20]) / 20
print(f"  Cached {len(ref_cache)} pairs")
print(f"  Avg ref margin (per-token): {avg_margin:.4f}")
print(f"  (Should be small and close to 0, not -19 like before)")
 
del ref_model, base
torch.cuda.empty_cache()
 
 
# ============================================================
# RUN ALL 3 EXPERIMENTS
# ============================================================
 
adapter_paths = {}
for exp in EXPERIMENTS:
    adapter_paths[exp] = run_experiment(exp, train_pairs, valid_pairs, ref_train, ref_valid)
 
 
# ============================================================
# TESTING
# ============================================================
 
print("\n\n" + "#" * 60)
print("  TESTING ALL ADAPTERS")
print("#" * 60)
 
section_kw = ["Item 1A", "Item 1", "Item 7", "Risk Factors", "MD&A", "Management's Discussion", "risk factors"]
judgment_kw = ["significant", "critical", "notable", "concern", "impact", "important", "material", "substantial"]
severity_kw = ["high", "medium", "low", "severe", "critical", "significant", "moderate"]
section_kw_ex = section_kw + ["10-K", "10-Q", "filing", "annual report", "disclosure", "SEC"]
change_kw = ["added", "removed", "modified", "changed", "new", "deleted", "updated", "introduced"]
time_kw = ["2024", "2025", "year-over-year", "prior year", "previous"]
 
 
def load_and_test(adapter_path, label):
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        trust_remote_code=True, device_map={"": 0},
    )
    model = PeftModel.from_pretrained(base, adapter_path)
    model.eval()
 
    def gen(msgs, max_tok=500):
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=max_tok, do_sample=False,
                                  pad_token_id=tokenizer.pad_token_id)
        return tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
 
    results = {}
    def rec(cat, name, ok):
        if cat not in results: results[cat] = []
        results[cat].append((name, ok))
 
    # Router (5)
    for exp, q in [("FILING_ANALYSIS","What are Apple's risk factors in their latest 10-K?"),
                    ("FILING_DIFF","How did Tesla's risks change from last year?"),
                    ("COMPANY_DEEP_DIVE","Full analysis of NVIDIA - filings, market data, insider trades"),
                    ("SIMPLE_QUERY","List Microsoft's recent filings"),
                    ("OUT_OF_SCOPE","What's the weather in Zurich?")]:
        r = gen([{"role":"system","content":ROUTER_SYSTEM_PROMPT},{"role":"user","content":q}], 20)
        rec("Router", exp, exp in r)
 
    # Tool calling (6)
    for q, tool, tick in [("What are Apple's risk factors?","get_filing_section","AAPL"),
                           ("How did Microsoft's MD&A change from last year?","diff_filing_sections","MSFT"),
                           ("List Tesla's recent 10-K filings","search_filings","TSLA"),
                           ("What are Tesla's risk factors?","get_filing_section","TSLA"),
                           ("What's the analyst consensus on NVIDIA?","analyst_ratings","NVDA"),
                           ("Show me recent insider trades at Meta","insider_trades","META")]:
        r = gen([{"role":"system","content":TOOL_SYSTEM_PROMPT},{"role":"user","content":q}], 100)
        fl = r.strip().split('\n')[0]
        try:
            p = json.loads(fl)
            ok = p.get("tool")==tool and p.get("arguments",{}).get("ticker")==tick
        except: ok = False
        rec("Tool calling", f"{tool}({tick})", ok)
 
    # Tool parsing (2)
    for nm, inp, ck in [
        ("AAPL", json.dumps({"ticker":"AAPL","section":"Risk Factors","filing_date":"2025-10-31",
                              "content":"International sales ~60%. Supply chain risks."}),
         {"t":["AAPL","Apple"],"s":["Risk","Item 1A"],"d":["2025","October"]}),
        ("MSFT", json.dumps({"ticker":"MSFT","section":"MD&A","filing_date":"2025-07-30",
                              "content":"Revenue +15% to $245.1B. Azure +33%."}),
         {"t":["MSFT","Microsoft"],"s":["MD&A","Management"],"d":["2025","July"]})]:
        r = gen([{"role":"system","content":SYSTEM_PROMPT},
                  {"role":"user","content":f"Analyze this filing section:\n{inp}"}], 500)
        ok = all(any(x in r for x in ck[k]) for k in ck)
        rec("Tool parsing", nm, ok)
 
    # Review (3)
    for nm, inp, exp in [
        ("Good","Original question: Apple risk factors?\n\nAnalysis: Apple's 10-K (Item 1A) identifies critical risks. International sales 60% ($234.3B). Gross margin 45.96%. Supply chain China 85%. R&D $29.9B.","COMPLETE"),
        ("Bad","Original question: Apple risk factors?\n\nAnalysis: Apple faces some risks. Competitive industry.","NEEDS_FOLLOW_UP"),
        ("Medium","Original question: Apple risk factors?\n\nAnalysis: Apple's 10-K Item 1A mentions supply chain and competition.","NEEDS_FOLLOW_UP")]:
        r = gen([{"role":"system","content":REVIEW_SYSTEM_PROMPT},{"role":"user","content":inp}], 100)
        ok = r.startswith("COMPLETE") if exp=="COMPLETE" else "NEEDS_FOLLOW_UP" in r
        rec("Review", nm, ok)
 
    # Synthesis (1)
    sd = json.dumps({"ticker":"DIS","metrics":{"market_cap":188.51e9,"pe_ratio":15.66},
                      "news":[{"headline":"Disney+ 150M subs"}],
                      "insider_trades":[{"name":"Bob Iger","type":"Sale"}]})
    r = gen([{"role":"system","content":SYSTEM_PROMPT},
              {"role":"user","content":f"Analyze DIS:\n{sd}"}], 500)
    ok = ("DIS" in r or "Disney" in r) and any(n in r for n in ["188","15.66","150"])
    rec("Synthesis", "DIS", ok)
 
    # Section summary (2)
    for q, t, n in [("What are the key risk factors in Apple's most recent 10-K?","AAPL","Apple"),
                     ("Summarize NVIDIA's MD&A from their latest 10-K.","NVDA","NVIDIA")]:
        r = gen([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":q}], 500)
        ok = (any(k.lower() in r.lower() for k in section_kw) and (t in r or n in r)
              and any(k in r.lower() for k in judgment_kw))
        rec("Section summary", t, ok)
 
    # Red flag (2)
    for q, t in [("What risks or concerns in Tesla's 10-K filing?","TSLA"),
                  ("Identify warning signs in Meta's annual filing.","META")]:
        r = gen([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":q}], 500)
        ok = len(r)>100 and any(k in r.lower() for k in severity_kw) and any(k.lower() in r.lower() for k in section_kw_ex)
        rec("Red flag", t, ok)
 
    # Change detection (2)
    for q, t in [("Changes between Google's 2024 and 2025 10-K risk factors?","GOOGL"),
                  ("How has Amazon's MD&A changed from 2024 to 2025 10-K?","AMZN")]:
        r = gen([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":q}], 500)
        ok = (any(k in r.lower() for k in change_kw) and any(k in r for k in time_kw)
              and any(k in r.lower() for k in judgment_kw))
        rec("Change detection", t, ok)
 
    total_p = sum(sum(1 for _,ok in v if ok) for v in results.values())
    total_c = sum(len(v) for v in results.values())
    scores = {}
    for cat in ["Router","Tool calling","Tool parsing","Review","Synthesis",
                "Section summary","Red flag","Change detection"]:
        if cat in results:
            p = sum(1 for _,ok in results[cat] if ok)
            scores[cat] = (p, len(results[cat]))
 
    del model, base
    torch.cuda.empty_cache()
    return scores, total_p, total_c
 
 
# Run tests
all_scores = {}
 
print("\n  Testing SFT...")
all_scores["SFT"], sft_p, sft_c = load_and_test(SFT_ADAPTER, "SFT")
 
for exp in EXPERIMENTS:
    print(f"\n  Testing {exp.upper()}...")
    all_scores[exp.upper()], _, _ = load_and_test(adapter_paths[exp], exp.upper())
 
 
# ============================================================
# COMPARISON TABLE
# ============================================================
 
print("\n\n" + "#" * 70)
print("  ABLATION RESULTS: SFT vs DPO vs IPO vs cDPO")
print(f"  Config: β={BETA}, SFT weight={SFT_WEIGHT}, {EPOCHS} epoch")
print(f"  Fixes: length-normalized log probs, early stop after {MIN_STEPS_BEFORE_EARLY_STOP} steps")
print("#" * 70)
 
cats = ["Router","Tool calling","Tool parsing","Review","Synthesis",
        "Section summary","Red flag","Change detection"]
labels = ["SFT"] + [e.upper() for e in EXPERIMENTS]
 
header = f"  {'Category':<20}" + "".join(f"{l:<10}" for l in labels)
print(header)
print(f"  {'-'*20}" + "-"*10*len(labels))
 
totals = {l: 0 for l in labels}
for cat in cats:
    row = f"  {cat:<20}"
    for l in labels:
        if cat in all_scores.get(l, {}):
            p, c = all_scores[l][cat]
            row += f"{p}/{c:<8}"
            totals[l] += p
        else:
            row += f"{'?':<10}"
    print(row)
 
print(f"  {'-'*20}" + "-"*10*len(labels))
row = f"  {'TOTAL':<20}"
for l in labels:
    row += f"{totals[l]}/23{'':5}"
print(row)
print("#" * 70)
 
# Save results
with open("/kaggle/working/ablation_results.txt", "w") as f:
    f.write(f"DPO Ablation Results\n{'='*60}\n")
    f.write(f"Config: β={BETA}, SFT weight={SFT_WEIGHT}, epochs={EPOCHS}\n")
    f.write(f"Fixes: length-normalized log probs, IPO margin clipping, min {MIN_STEPS_BEFORE_EARLY_STOP} steps\n\n")
    for l in labels:
        f.write(f"\n{l}:\n")
        for cat in cats:
            if cat in all_scores.get(l, {}):
                p, c = all_scores[l][cat]
                f.write(f"  {cat}: {p}/{c}\n")
        f.write(f"  Total: {totals[l]}/23\n")
 
# Copy best adapter
best_label = max(EXPERIMENTS, key=lambda e: totals[e.upper()])
shutil.copytree(adapter_paths[best_label], "/kaggle/working/best_dpo_adapter", dirs_exist_ok=True)
print(f"\n  Best: {best_label.upper()} ({totals[best_label.upper()]}/23)")
print("  Results saved to /kaggle/working/ablation_results.txt")
print("  Best adapter saved to /kaggle/working/best_dpo_adapter/")